# Lab 1  Regular Expressions

In [1]:
import re

## Exercise 1  Literal matching and counting

In [2]:
text = "NLP is useful. NLP is interesting. Machine learning is useful."

# Q1: what each pattern finds
print(re.findall(r"NLP", text))       # 2 matches
print(re.findall(r"useful", text))    # 2 matches

# 2: NLP only as a whole word
print(re.findall(r"\bNLP\b", text))
print(re.findall(r"\bNLP\b", "NLPs is useful. NLP is interesting."))  # NLPs is NOT matched

# 3: generic counter + alternation
def count_occurrences(pattern, text, flags=0):
    return len(re.findall(pattern, text, flags))

print(count_occurrences(r"useful|interesting", text))  # 3

# 4: case-insensitive
# NOTE: the line `re.I = RegexFlag.NOFLAG` from the original file was removed:
# it overwrote re.I with 0, which silently disables case-insensitivity.
mixed = "NLP is useful. nlp is interesting. Nlp is fun. NLPs are many."
print(re.findall(r"\bNLP\b", mixed))                 # only 'NLP'
print(re.findall(r"\bNLP\b", mixed, re.IGNORECASE))  # 'NLP', 'nlp', 'Nlp'

# Challenge: % of sentences mentioning NLP or machine learning
sentence_pattern = r"[^.]*\b(?:NLP|machine learning)\b[^.]*\."
hits = re.findall(sentence_pattern, text, re.IGNORECASE)
print(len(hits), "/ 3 ->", round(100 * len(hits) / 3, 1), "%")

['NLP', 'NLP']
['useful', 'useful']
['NLP', 'NLP']
['NLP']
3
['NLP']
['NLP', 'nlp', 'Nlp']
3 / 3 -> 100.0 %


## Exercise 2  Character classes and a simple validator

In [3]:
text = "cyt ybyty yrat dog hot"

# 1: words ending in 'at' whatever the preceding letter
print(re.findall(r"\b\w+at\b", "cat bat rat dog hot"))
print(re.findall(r"\b\w*at\b", text))

# 2: 3-letter words with consonants only (y counted as a consonant)
print(re.findall(r"\b[b-df-hj-np-tv-z]{3}\b", text, re.IGNORECASE))

# 3 + 4: username validator
def check_username(u):
    if len(u) < 4:
        return "INVALID (too short)"
    if len(u) > 12:
        return "INVALID (too long)"
    if not re.fullmatch(r"[a-z0-9_]+", u):
        return "INVALID (forbidden character)"
    return "VALID"

# one-regex version: [a-z0-9_]{4,12}
for u in ['ali_92', 'A', 'student.name', 'nlp_lab_2026']:
    print(f"{u!r:16} -> {check_username(u)}  | regex ok: {bool(re.fullmatch(r'[a-z0-9_]{4,12}', u))}")

['cat', 'bat', 'rat']
['yrat']
['cyt']
'ali_92'         -> VALID  | regex ok: True
'A'              -> INVALID (too short)  | regex ok: False
'student.name'   -> INVALID (forbidden character)  | regex ok: False
'nlp_lab_2026'   -> VALID  | regex ok: True


## Exercise 3  Digits, decimals and prices

In [4]:
text = "The course has 20 hours, 3 labs and 12 exercises."
print(re.findall(r"\d+", text))

receipt = "2 apples, 1.5 kg of rice, 3 juices at $2.50 each, total: $12.75 (-$1.00 discount)."

# 1: decimals
print(re.findall(r"\d+(?:\.\d+)?", receipt))

# 2: negative amounts (optional minus, optional $)
print(re.findall(r"-?\$?\d+(?:\.\d+)?", receipt))

# 3: only amounts preceded by $, as floats (sign kept)
amounts = [float(sign + num) for sign, num in re.findall(r"(-?)\$(\d+(?:\.\d+)?)", receipt)]
print(amounts)

# 4: sum
print(sum(amounts))  # 2.5 + 12.75 - 1.0 = 14.25

# Challenge: quantities (not attached to $) vs monetary amounts
quantities = re.findall(r"(?<![\$\d.])\d+(?:\.\d+)?", receipt)
money = re.findall(r"-?\$\d+(?:\.\d+)?", receipt)
print(quantities)
print(money)

['20', '3', '12']
['2', '1.5', '3', '2.50', '12.75', '1.00']
['2', '1.5', '3', '$2.50', '$12.75', '-$1.00']
[2.5, 12.75, -1.0]
14.25
['2', '1.5', '3']
['$2.50', '$12.75', '-$1.00']


## Exercise 4  Dates and times in mixed formats

In [5]:
text = "The exam is on 16/09/2026 at 14:30."
print(re.findall(r"\d{2}/\d{2}/\d{4}", text))
print(re.findall(r"\d{2}:\d{2}", text))

schedule = ("Lab 1: 16/09/2026, Lab 2: 2026-09-23, Lab 3: 30-09-2026, "
            "exam at 9:00, revision session at 14:30, office hours 09h30.")

# 1: DD/MM/YYYY and DD-MM-YYYY
print(re.findall(r"\b\d{2}[/-]\d{2}[/-]\d{4}\b", schedule))

# 2: ISO YYYY-MM-DD
print(re.findall(r"\b\d{4}-\d{2}-\d{2}\b", schedule))

# 3: all three at once
date_pattern = r"\b(?:\d{2}[/-]\d{2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b"
print(re.findall(date_pattern, schedule))

# 4: times 9:00 / 14:30 / 09h30
# change: separator class [:h] and hour made 1-2 digits (\d{1,2})
time_pattern = r"\b\d{1,2}[:h]\d{2}\b"
print(re.findall(time_pattern, schedule))

# Q2: one-digit day/month -> use \d{1,2}
print(re.findall(r"\b\d{1,2}/\d{1,2}/\d{4}\b", "6/9/2026"))
# Q3: the regex only checks the shape: 31/02/2026 matches but is not a real date
print(re.findall(r"\b\d{2}/\d{2}/\d{4}\b", "31/02/2026"))

['16/09/2026']
['14:30']
['16/09/2026', '30-09-2026']
['2026-09-23']
['16/09/2026', '2026-09-23', '30-09-2026']
['9:00', '14:30', '09h30']
['6/9/2026']
['31/02/2026']


## Exercise 5  Email addresses and a validity checker

In [6]:
text = "Contact ali@example.com or student.name@univ.dz."
pattern = r"[\w.-]+@[\w.-]+\.\w+"
print(re.findall(pattern, text))

mailing = ['ali@example.com', 'student.name@univ.dz', 'a.b-c@sub.univ.dz',
           'not-an-email', 'missing@domain', 'double@@example.com',
           '@no-local-part.com', 'ok.name+lab@example.co.uk']

# 1: original pattern, whole-entry match
for e in mailing:
    print(f"{e:28} original: {bool(re.fullmatch(pattern, e))}")

# Tasks 2-3: improved pattern ('+' added to the local-part class, domain = labels + dots, TLD >= 2 letters)
email_pattern = r"[\w.+-]+@(?:[\w-]+\.)+[A-Za-z]{2,}"

def is_valid_email(address):
    return re.fullmatch(email_pattern, address) is not None

# 4: report
valid = [e for e in mailing if is_valid_email(e)]
invalid = [e for e in mailing if not is_valid_email(e)]
print("Valid:", len(valid), valid)
print("Invalid:", len(invalid), invalid)

['ali@example.com', 'student.name@univ.dz']
ali@example.com              original: True
student.name@univ.dz         original: True
a.b-c@sub.univ.dz            original: True
not-an-email                 original: False
missing@domain               original: False
double@@example.com          original: False
@no-local-part.com           original: False
ok.name+lab@example.co.uk    original: False
Valid: 4 ['ali@example.com', 'student.name@univ.dz', 'a.b-c@sub.univ.dz', 'ok.name+lab@example.co.uk']
Invalid: 4 ['not-an-email', 'missing@domain', 'double@@example.com', '@no-local-part.com']


## Exercise 6  URLs and domain extraction

In [7]:
text = "Visit https://www.univ.dz or http://example.org."
print(re.findall(r"https?://[^\s]+", text))  # trailing '.' captured

paragraph = ("See https://www.univ.dz/lab1?page=2, http://example.org, "
             "https://docs.python.org/3/library/re.html and ftp://old-server.com "
             "(not a web link).")

# 1: original vs fixed (URL must end with a word char or '/')
print(re.findall(r"https?://[^\s]+", paragraph))
url_pattern = r"https?://[^\s]*[\w/]"
urls = re.findall(url_pattern, paragraph)
print(urls)

# 2: domain, applied to extracted URLs
print([re.search(r"://([^/\s?#:]+)", u).group(1) for u in urls])

# 3: ftp is ignored because the pattern requires 'http' + optional 's'. Adapted:
url_pattern_ftp = r"(?:https?|ftp)://[^\s]*[\w/]"
print(re.findall(url_pattern_ftp, paragraph))

# 4: HTTPS vs HTTP
n_https = len([u for u in urls if u.startswith("https://")])
n_http = len([u for u in urls if u.startswith("http://")])
print("https:", n_https, "http:", n_http)

['https://www.univ.dz', 'http://example.org.']
['https://www.univ.dz/lab1?page=2,', 'http://example.org,', 'https://docs.python.org/3/library/re.html']
['https://www.univ.dz/lab1?page=2', 'http://example.org', 'https://docs.python.org/3/library/re.html']
['www.univ.dz', 'example.org', 'docs.python.org']
['https://www.univ.dz/lab1?page=2', 'http://example.org', 'https://docs.python.org/3/library/re.html', 'ftp://old-server.com']
https: 2 http: 1


## Exercise 7  Substitution and a mini PII-redaction tool

In [8]:
text = "The course has 20 hours and 12 exercises."
# 1
print(re.sub(r"\d+", "<NUMBER>", text))

comment = ("Contact me at ali.ben@example.com or call 0555 12 34 56 if you have "
           "questions about the 20-hour course.  Ali, 16/09/2026.")

EMAIL = r"[\w.+-]+@(?:[\w-]+\.)+[A-Za-z]{2,}"
DATE = r"\b(?:\d{1,2}[/-]\d{1,2}[/-]\d{4}|\d{4}-\d{2}-\d{2})\b"
# 0XXX XX XX XX / 0XXXXXXXXX / +213 XXX XX XX XX  (challenge: several formats)
PHONE = r"(?<!\d)(?:\+213|0)\s?[5-7]\d{2}(?:[\s.]?\d{2}){3}(?!\d)"

def anonymize(text):
    text = re.sub(EMAIL, "<EMAIL>", text)
    text = re.sub(DATE, "<DATE>", text)    # dates before phones so digits aren't confused
    text = re.sub(PHONE, "<PHONE>", text)
    return text

print(anonymize(comment))  # 'course' and '20' are untouched
for p in ["0555123456", "0555 12 34 56", "+213 555 12 34 56"]:
    print(p, "->", anonymize(p))

The course has <NUMBER> hours and <NUMBER> exercises.
Contact me at <EMAIL> or call <PHONE> if you have questions about the 20-hour course.  Ali, <DATE>.
0555123456 -> <PHONE>
0555 12 34 56 -> <PHONE>
+213 555 12 34 56 -> <PHONE>


## Exercise 8  RegEx analysis and limitations

In [9]:
sentence = "Dr. O'Brien met 3 students in New- York on 16/09/2026."

# 1. \d+          : one or more digits ('42', '2026')         | fails: '1.5' -> ['1','5'], '²' etc.
# 2. [A-Z][a-z]+  : capital + lowercase letters ('Paris','Ali') | fails: 'NLP' (acronym), 'McDonald', 'Élodie'
# 3. \b\w+\b      : a "word" ('hello','x2')                    | fails: 'well-known' -> 2 words, "don't" -> 'don','t'
for p in [r"\d+", r"[A-Z][a-z]+", r"\b\w+\b"]:
    print(p, "->", re.findall(p, sentence))

# Unexpected results on the sentence:
# \d+          -> date split into '16','09','2026' (no notion of a date)
# [A-Z][a-z]+  -> 'Dr' (the '.' is dropped), "O'Brien" becomes only 'Brien' (the 'O' is lost),
#                 'New' and 'York' split by the broken hyphen
# \b\w+\b      -> "O'Brien" -> 'O','Brien'; 'New-' / 'York' split; the date is split in 3 numbers
# Fragility: contractions (don't), hyphenation (New-York), acronyms (NLP, U.S.A.),
# multi-word names (New York), abbreviations with dots (Dr.), accents/non-Latin scripts.

\d+ -> ['3', '16', '09', '2026']
[A-Z][a-z]+ -> ['Dr', 'Brien', 'New', 'York']
\b\w+\b -> ['Dr', 'O', 'Brien', 'met', '3', 'students', 'in', 'New', 'York', 'on', '16', '09', '2026']


In [11]:
announcement = ("Reminder: the NLP project (worth 12 points) is due on 30/09/2026 before 23:59. "
                "Send your report (max 2.5 MB) to nlp.lab@univ.dz or upload it at "
                "https://courses.univ.dz/submit. Late submissions lose -1.5 points per day. "
                "Contact +213 555 98 76 54 for questions.")

TIME = r"\b\d{1,2}[:h]\d{2}\b"
URL = r"(?:https?|ftp)://[^\s]*[\w/]"

results = {
    "dates": re.findall(DATE, announcement),
    "times": re.findall(TIME, announcement),
    "emails": re.findall(EMAIL, announcement),
    "urls": re.findall(URL, announcement),
    "phones": re.findall(PHONE, announcement),
}

# numbers: remove everything already extracted so their digits aren't counted again
remaining = announcement
for pat in (EMAIL, URL, DATE, TIME, PHONE):
    remaining = re.sub(pat, " ", remaining)
results["numbers"] = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", remaining)]

for category, values in results.items():
    print(f"{category:8}: {values}")

# Discussion:
# Easiest: emails and URLs (very distinctive delimiters '@' and '://').
# Hardest: numbers (ambiguous: part of a date, phone, time, version... so they must be
# extracted last, after masking the other categories) and phones (many formats).

dates   : ['30/09/2026']
times   : ['23:59']
emails  : ['nlp.lab@univ.dz']
urls    : ['https://courses.univ.dz/submit']
phones  : ['+213 555 98 76 54']
numbers : [12.0, 2.5, -1.5]
